# Solutions · 19 · Sparse matrices and 2D PDEs at realistic size

Worked solutions to the exercises of [notebook 19](../notebooks/19_sparse_2d_pdes.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engmath is missing): install it from GitHub
    import engmath
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmath"], check=True)
import numpy as np
import matplotlib.pyplot as plt
import engmath
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")
import time
import scipy.sparse as sps
import scipy.sparse.linalg as spla
from engmath import sparse
Lx, Ly, thick, kb, P = 0.100, 0.060, 1.6e-3, 20.0, 2.0
q_vol = P/(0.02*0.02*thick)
def source(x, y):
    return np.where((np.abs(x - Lx/2) <= 0.01) & (np.abs(y - Ly/2) <= 0.01), q_vol/kb, 0.0)
def grid(nx, ny):
    X, Y = np.meshgrid(np.linspace(0, Lx, nx + 2)[1:-1], np.linspace(0, Ly, ny + 2)[1:-1])
    return source(X, Y).ravel()

## Exercise 1

The board also loses heat from both faces by convection, $h$ = 10 W/(m²·K). This adds a term
   $(2h/(k\,t))\,\theta$ to the equation, i.e. $A \to A + \frac{2h}{k t} I$. How much does the peak temperature drop?

In [2]:
nx, ny = 199, 119
A = sparse.laplacian_2d(nx, ny, Lx/(nx + 1), Ly/(ny + 1))
f = grid(nx, ny)
theta_no = spla.spsolve(A.tocsc(), f)
h_conv = 10.0
A_conv = A + (2*h_conv/(kb*thick)) * sps.identity(nx*ny)
theta_conv = spla.spsolve(A_conv.tocsc(), f)
print(f"peak temperature: edges only {25 + theta_no.max():.1f} °C, with face convection {25 + theta_conv.max():.1f} °C")
print(f"heat leaving through the faces: {h_conv*2*np.sum(theta_conv)*(Lx/(nx+1))*(Ly/(ny+1)):.2f} W of {P} W")

peak temperature: edges only 41.8 °C, with face convection 40.1 °C
heat leaving through the faces: 0.39 W of 2.0 W


Convection from both faces removes heat everywhere, not only at the clamped edges, and lowers the peak
temperature. The term simply adds a constant to the diagonal - the matrix stays sparse, symmetric and
positive definite, so the same solvers apply. The energy balance shows how much of the 2 W now leaves
through the faces instead of the frame.

## Exercise 2

Time the direct solver for grids of 100², 200², 400² and 800² nodes (if memory allows). How does the
   time grow with the number of unknowns?

In [3]:
sizes, times = [], []
for n in (100, 200, 400):
    A = sparse.laplacian_2d(n, n, 1/(n + 1), 1/(n + 1)).tocsc()
    b = np.ones(n*n)
    t0 = time.perf_counter(); spla.spsolve(A, b); dt = time.perf_counter() - t0
    sizes.append(n*n); times.append(dt)
    print(f"{n}² = {n*n:>7,} unknowns: {dt:6.2f} s")
print(f"time grows like N^{np.polyfit(np.log(sizes), np.log(times), 1)[0]:.2f}")

100² =  10,000 unknowns:   0.03 s


200² =  40,000 unknowns:   0.18 s


400² = 160,000 unknowns:   1.07 s
time grows like N^1.33


The sparse direct solver's time grows faster than linearly in the number of unknowns $N$ (here about
$N^{1.3}$; theory for large 2D grids with a good ordering is about $N^{1.5}$), because factorisation creates
*fill-in*: new nonzeros
inside the band. It stays practical up to hundreds of thousands of 2D unknowns; for very large or 3D
problems, iterative methods with good preconditioners (conjugate gradient + multigrid) scale better.
An 800 × 800 grid (640,000 unknowns) is left to try on your own machine.

## Exercise 3

Is the 63 % time accurate? Repeat the transient with time steps of 2 s and 0.5 s. How large is the
   time-step error of backward Euler here, and how does it scale with the step?

In [4]:
rho_c = 1900*1100; alpha = kb/rho_c
nxt, nyt = 99, 59
At = sparse.laplacian_2d(nxt, nyt, Lx/(nxt + 1), Ly/(nyt + 1))
ft = grid(nxt, nyt)
theta_ss = spla.spsolve(At.tocsc(), ft).max()
def transient(dt, t_end=60.0):
    lu = spla.splu((sps.identity(nxt*nyt, format="csc") + dt*alpha*At).tocsc())
    theta = np.zeros(nxt*nyt); times, peak = [0.0], [0.0]
    for step in range(int(round(t_end/dt))):
        theta = lu.solve(theta + dt*alpha*ft)
        times.append((step + 1)*dt); peak.append(theta.max())
    times, peak = np.array(times), np.array(peak)
    t63 = np.interp(0.632*theta_ss, peak, times)        # interpolate between steps
    return t63, np.interp(10.0, times, peak)
ref = transient(0.05)
for dt in (2.0, 1.0, 0.5):
    t63, p10 = transient(dt)
    print(f"dt = {dt:3.1f} s: 63 % time {t63:6.2f} s (error {t63 - ref[0]:+.2f} s); rise after 10 s {p10:.3f} K (error {p10 - ref[1]:+.3f} K)")
print(f"reference (dt = 0.05 s): 63 % time {ref[0]:.2f} s")

dt = 2.0 s: 63 % time  18.05 s (error +0.94 s); rise after 10 s 7.930 K (error -0.338 K)
dt = 1.0 s: 63 % time  17.57 s (error +0.46 s); rise after 10 s 8.101 K (error -0.167 K)
dt = 0.5 s: 63 % time  17.33 s (error +0.22 s); rise after 10 s 8.189 K (error -0.079 K)
reference (dt = 0.05 s): 63 % time 17.11 s


Backward Euler is first order in time: halving the step roughly halves the error. The errors are also
systematic - backward Euler is overly damped, so the board appears to heat up *more slowly* than it
really does. Reading the 63 % time off the step grid adds a further error of up to one step, which is why
it is interpolated here.

## Exercise 4

**Implement it yourself:** add a Jacobi preconditioner to `conjugate_gradient` (divide the residual by the diagonal of $A$ when forming the search direction). How many iterations does it save for the board problem? Why so few, for this particular matrix?

In [5]:
def pcg(A, b, M_inv_diag, tol=1e-10):
    x = np.zeros_like(b); r = b.copy(); z = M_inv_diag*r; p = z.copy(); rz = r @ z
    for it in range(1, 10*b.size):
        Ap = A @ p; a = rz/(p @ Ap); x += a*p; r -= a*Ap
        if np.linalg.norm(r)/np.linalg.norm(b) < tol:
            return x, it
        z = M_inv_diag*r; rz_new = r @ z; p = z + (rz_new/rz)*p; rz = rz_new
nx, ny = 299, 179
A = sparse.laplacian_2d(nx, ny, Lx/(nx + 1), Ly/(ny + 1))
b = grid(nx, ny)
_, hist = sparse.conjugate_gradient(A, b, tol=1e-10)
_, it_j = pcg(A, b, 1/A.diagonal())
print(f"plain CG: {len(hist) - 1} iterations; Jacobi-preconditioned CG: {it_j} iterations")
print(f"diagonal of A: min {A.diagonal().min():.4g}, max {A.diagonal().max():.4g}")

plain CG: 937 iterations; Jacobi-preconditioned CG: 937 iterations
diagonal of A: min 3.6e+07, max 3.6e+07


No saving at all: for this matrix every diagonal entry is the same, so the Jacobi preconditioner merely
rescales the problem by a constant and CG takes the same steps. Jacobi helps when the diagonal varies a
lot (e.g. strongly varying conductivity or cell sizes). For the Laplacian itself, effective
preconditioners must capture the coupling between nodes - incomplete Cholesky factorisation or multigrid.